In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
# return_X_y=True: trả thẳng về X và y, gọn hơn cách lấy data.data, data.target ở Buổi 1.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)   # (426, 30) (143, 30)

(426, 30) (143, 30)


In [3]:
# Ô 3 — Cách 1: tách thêm một tập validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train,y_train, test_size=0.25, stratify=y_train, random_state=42)
print(X_tr.shape, X_val.shape, X_test.shape)

scores = {}

for d in range(1,9) :
  t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr,y_tr)
  scores[d] = round(t.score(X_val,y_val),3)
print(scores)

best_d = max(scores, key=scores.get)
# max(scores, key=scores.get): tìm key có value lớn nhất. key=scores.get bảo max so sánh theo điểm thay vì theo độ sâu. Khi hòa điểm (2, 6, 7 cùng 0,935), max lấy key gặp đầu tiên là 2.
print(best_d)

(319, 30) (107, 30) (143, 30)
{1: 0.869, 2: 0.935, 3: 0.925, 4: 0.916, 5: 0.907, 6: 0.935, 7: 0.935, 8: 0.888}
2


In [4]:
for rs in [0, 1, 2, 3, 4]:
    a, b, c, d_ = train_test_split(X_train, y_train, test_size=0.25,
                                   stratify=y_train, random_state=rs)
    # random_state từ 0 đến 4, tức 5 cách chia validation khác nhau.
    sc = [DecisionTreeClassifier(max_depth=d, random_state=42).fit(a, c).score(b, d_)
          for d in range(1, 9)]
    print(sc)
    print(rs, "độ sâu tốt nhất:", int(np.argmax(sc)) + 1, "điểm:", round(max(sc), 3))
    # np.argmax(sc): vị trí của điểm lớn nhất trong list (bắt đầu từ 0)

[0.9158878504672897, 0.9345794392523364, 0.9719626168224299, 0.9719626168224299, 0.9439252336448598, 0.9532710280373832, 0.9626168224299065, 0.9532710280373832]
0 độ sâu tốt nhất: 3 điểm: 0.972
[0.897196261682243, 0.9439252336448598, 0.9439252336448598, 0.9345794392523364, 0.9345794392523364, 0.9439252336448598, 0.9252336448598131, 0.9252336448598131]
1 độ sâu tốt nhất: 2 điểm: 0.944
[0.8785046728971962, 0.9532710280373832, 0.9345794392523364, 0.9345794392523364, 0.9439252336448598, 0.9439252336448598, 0.9439252336448598, 0.9439252336448598]
2 độ sâu tốt nhất: 2 điểm: 0.953
[0.8785046728971962, 0.9158878504672897, 0.897196261682243, 0.9065420560747663, 0.897196261682243, 0.9065420560747663, 0.9065420560747663, 0.9065420560747663]
3 độ sâu tốt nhất: 2 điểm: 0.916
[0.8691588785046729, 0.9065420560747663, 0.9065420560747663, 0.9158878504672897, 0.9252336448598131, 0.9252336448598131, 0.9252336448598131, 0.9252336448598131]
4 độ sâu tốt nhất: 5 điểm: 0.925


In [7]:
# Ô 5 — Cách 2: cross-validation

cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

for i, (tr_idx, va_idx) in enumerate(cv.split(X_train,y_train)) :
  #  print(va_idx)
  print(f"Fold {i}: học {len(tr_idx)} mẫu, thi thử {len(va_idx)} mẫu, "
        f"tỷ lệ lành tính {y_train.iloc[va_idx].mean():.3f}") 

print("--------")

tree3 = DecisionTreeClassifier(max_depth=3,random_state=42)
s = cross_val_score(tree3,X_train,y_train,cv=cv)
print(s.round(3))
print(round(s.mean(),3) , "±" , round(s.std(),3))


Fold 0: học 340 mẫu, thi thử 86 mẫu, tỷ lệ lành tính 0.628
Fold 1: học 341 mẫu, thi thử 85 mẫu, tỷ lệ lành tính 0.624
Fold 2: học 341 mẫu, thi thử 85 mẫu, tỷ lệ lành tính 0.624
Fold 3: học 341 mẫu, thi thử 85 mẫu, tỷ lệ lành tính 0.624
Fold 4: học 341 mẫu, thi thử 85 mẫu, tỷ lệ lành tính 0.635
--------
[0.942 0.918 0.941 0.918 0.976]
0.939 ± 0.022


In [ ]:
# Ô 6 — Baseline: mô hình “lười” được bao nhiêu điểm
dummy = DummyClassifier(strategy="most_frequent") # không nhìn feature, luôn đoán lớp đông nhất (lành tính).
d = cross_val_score(dummy,X_train, y_train, cv=cv)
print(d.round(3), round(d.mean(),3))

#Vì sao phải có baseline: nếu không biết mốc này, nghe “cây đạt 93,9%” bạn không biết là giỏi hay chỉ hơn đoán mò một chút. 
#Ở đây cây hơn baseline khoảng 31 điểm, rõ ràng đã học được nhiều. Với dữ liệu mất cân bằng nặng như M3, 
# baseline có thể rất cao, nên bước này càng quan trọng.

[0.628 0.624 0.624 0.624 0.635] 0.627


In [11]:
# Ô 7 — Chọn độ sâu bằng cross-validation
rows = []
for d in range(1,9) : 
  s = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42),X_train, y_train, cv=cv)
  rows.append({"max_depth": d, "mean": round(s.mean(), 3), "std": round(s.std(), 3)})
print(pd.DataFrame(rows).to_string(index=False))


 max_depth  mean   std
         1 0.892 0.019
         2 0.925 0.035
         3 0.939 0.022
         4 0.927 0.019
         5 0.927 0.022
         6 0.923 0.018
         7 0.925 0.016
         8 0.925 0.016


In [ ]:
# Ô 8 — Logistic Regression hoạt động thế nào
# Logistic Regression tính z = b + w1·x1 + w2·x2 + ...: mỗi feature x nhân với một hệ số w, cộng thêm hằng số b.
for z in [-4, -2, 0, 2 , 4] :
  print(z, round(1 / (1+ np.exp(-z)),3))
  # 1 / (1 + e^(-z)) là hàm sigmoid, ép z (từ âm vô cùng đến dương vô cùng) vào khoảng 0 đến 1. z = 0 cho xác suất 0,5; z càng dương thì càng gần 1.

print("-----------")
# Bước 2: vì sao cần StandardScaler.
print(X_train[["mean area", "mean smoothness"]].describe().loc[["mean", "std"]].round(3))
# Hai feature chênh nhau khoảng 10.000 lần về độ lớn. Logistic Regression học tốt hơn khi mọi feature cùng thang đo. 
# StandardScaler đưa mỗi feature về trung bình 0 và độ lệch chuẩn 1,
print("------------")

# Bước 3: huấn luyện và đọc hệ số.
lr = make_pipeline(StandardScaler(), LogisticRegression()) # ghép hai bước thành một mô hình
s = cross_val_score(lr, X_train, y_train, cv=cv)
print(round(s.mean(), 3), "±", round(s.std(), 3))

lr.fit(X_train, y_train) # Khi fit, dữ liệu được chuẩn hóa rồi mới đưa vào Logistic Regression
# lr[-1]: lấy bước cuối cùng của pipeline, tức Logistic Regression. .coef_[0] là 30 hệ số.
# pd.Series(..., index=X.columns).sort_values(): gắn tên feature vào từng hệ số rồi sắp xếp.
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values()
print(coef.head(3).round(2))

x0 = X_test.iloc[[0]] # hai lớp ngoặc giữ kết quả ở dạng bảng một dòng
z = lr.decision_function(x0)[0]
print(round(z, 2), round(1 / (1 + np.exp(-z)), 3), lr.predict_proba(x0).round(3))


-4 0.018
-2 0.119
0 0.5
2 0.881
4 0.982
-----------
      mean area  mean smoothness
mean    649.627            0.096
std     345.940            0.013
------------
0.979 ± 0.009
worst texture    -1.25
radius error     -1.07
worst symmetry   -0.96
dtype: float64
3.44 0.969 [[0.031 0.969]]


In [ ]:
# Ô 9 — Siêu tham số C của Logistic Regression

# C là gì: núm vặn mức độ cho phép mô hình phức tạp.
# C nhỏ: mô hình bị “kìm” mạnh, hệ số bị ép về gần 0, nên quá đơn giản (underfitting).
# C lớn: mô hình được tự do khớp sát dữ liệu train, nên dễ học thuộc (overfitting).
rows = []
for C in [0.001, 0.01, 0.1, 1, 10, 100] :
  model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
  s = cross_val_score(model,X_train,y_train,cv=cv)
  train_acc= model.fit(X_train,y_train).score(X_train, y_train)
  rows.append({"C": C, "cv_mean": round(s.mean(), 3), "cv_std": round(s.std(), 3),
                 "train": round(train_acc, 3)})
print(pd.DataFrame(rows).to_string(index=False))

#         C  cv_mean  cv_std  train
#    0.001    0.880   0.038  0.892   <- quá đơn giản (underfitting)
#    0.010    0.951   0.029  0.958
#    0.100    0.977   0.013  0.984
#    1.000    0.979   0.009  0.988   <- tốt nhất
#   10.000    0.967   0.012  0.991
#  100.000    0.965   0.013  0.998   <- train gần 100%, CV giảm (overfitting)

[0.89534884 0.87058824 0.87058824 0.82352941 0.94117647]
[0.95348837 0.97647059 0.91764706 0.91764706 0.98823529]
[0.96511628 0.96470588 0.97647059 0.97647059 1.        ]
[0.97674419 0.96470588 0.97647059 0.98823529 0.98823529]
[0.96511628 0.95294118 0.98823529 0.96470588 0.96470588]
[0.95348837 0.96470588 0.98823529 0.95294118 0.96470588]
      C  cv_mean  cv_std  train
  0.001    0.880   0.038  0.892
  0.010    0.951   0.029  0.958
  0.100    0.977   0.013  0.984
  1.000    0.979   0.009  0.988
 10.000    0.967   0.012  0.991
100.000    0.965   0.013  0.998


In [ ]:
# Ô 10 — Mở “đề thi thật” đúng một lần
final_models = {
    "Dummy (lớp đông)": DummyClassifier(strategy="most_frequent"),
    "Cây, max_depth=3": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Logistic Regression, C=1": make_pipeline(StandardScaler(), LogisticRegression(C=1)),
}
for name, m in final_models.items():
    m.fit(X_train, y_train)
    print(f"{name:28s} test = {m.score(X_test, y_test):.3f}")
# Điểm test của mỗi mô hình nằm gần điểm cross-validation: cross-validation đã dự báo đúng điểm test mà không cần nhìn vào test. Đó là lý do nó đáng tin.

Dummy (lớp đông)             test = 0.629
Cây, max_depth=3             test = 0.944
Logistic Regression, C=1     test = 0.986


In [ ]:
# Ô 11 — Liên hệ M3: vì sao M3 cần cross-validation
from pathlib import Path
df = pd.read_csv(Path("..") / "data" / "raw" / "evtx_data.csv", low_memory=False)
files = df.groupby("EVTX_FileName")["EVTX_Tactic"].first().reset_index(name="tactic")

f_train, f_test = train_test_split(files, test_size=0.2, stratify=files["tactic"],
                                   random_state=42)
print(len(f_test), f_test["tactic"].value_counts().to_dict())
# 50 {'Privilege Escalation': 12, 'Lateral Movement': 10, ..., 'Discovery': 2,
#     'Command and Control': 1}